# 09 — Hyper-parameter tuning

**Purpose.** Random search for FedProx on a held-out validation split carved from the training data (the test set is never used for selection).

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/09_hyperparameter_tuning_*.png|pdf` and `reports/notebooks/09_hyperparameter_tuning.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "09_hyperparameter_tuning"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Random search with a validation split

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.preprocessing import split_validation
from federated_learning.data_partitioning import partition
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig
d = make_dataset(n_samples=6000, seed=SEED); tr = split_validation(d, .25, SEED)   # tr.X_test is the VALIDATION split carved from train
parts = partition("dirichlet", tr.y_train, 10, SEED, alpha=.5); rng = np.random.default_rng(SEED); rows = []
for t in range(8):
    p = dict(lr=float(10 ** rng.uniform(-2.5, -0.7)), mu=float(10 ** rng.uniform(-3, 0)), local_epochs=int(rng.integers(1, 4)))
    r = run_fl(FLConfig(algo="fedprox", rounds=10, seed=SEED, **p), tr, parts); rows.append(dict(**p, val_acc=r.final["acc"]))
tab = pd.DataFrame(rows).sort_values("val_acc", ascending=False).round(4); display(tab); best = tab.iloc[0]
SUMMARY["search"] = tab.to_dict("records")

,lr,mu,local_epochs,val_acc
2,0.1390,0.0661,3,0.8800
4,0.0930,0.0010,2,0.8200
3,0.0301,0.6386,3,0.7495
0,0.0443,0.0064,1,0.7267
7,0.0110,0.0185,2,0.6638
5,0.0036,0.1545,3,0.6162
6,0.0065,0.3886,1,0.6057
1,0.0034,0.2753,1,0.5676


## 2. Refit the best configuration on the full training split and evaluate once on test

In [3]:
parts_full = partition("dirichlet", d.y_train, 10, SEED, alpha=.5)
fin = run_fl(FLConfig(algo="fedprox", rounds=10, seed=SEED, lr=float(best.lr), mu=float(best.mu), local_epochs=int(best.local_epochs)), d, parts_full)
dflt = run_fl(FLConfig(algo="fedprox", rounds=10, seed=SEED), d, parts_full)
print("tuned test acc", round(fin.final["acc"], 4), "| default", round(dflt.final["acc"], 4)); SUMMARY["test"] = dict(tuned=fin.final["acc"], default=dflt.final["acc"])
fig, ax = plt.subplots(figsize=(4.4, 2.8)); ax.scatter(np.log10(tab.lr), tab.val_acc, c=np.log10(tab.mu), cmap="viridis"); ax.set_xlabel("log10 learning rate"); ax.set_ylabel("validation accuracy"); ax.set_title("Random search (colour = log10 mu)")
save_fig(fig, "search"); save_summary()

tuned test acc 0.9183 | default 0.8556
summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/09_hyperparameter_tuning.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.